# Laya × PDelta3-GDN2 + bidirectional Local32

[Open in Colab](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/codex/fix-laya-pdelta-local32/notebooks/Laya_PDelta3_GDN2_CLVR_Colab.ipynb)

Restores the working Qwen recipe's **query-dependent Local32 softmax branch** and low-rate QKV tuning, adapted to Laya's **bidirectional ModernBERT** encoder. The earlier Laya port had only a value convolution, which cannot reproduce query-dependent local attention.

This is a **hybrid**, not an attention-free model. GDN2 and learned global linear attention supply memory/global context. Laya routes the previous encoder representation; it does not reproduce Qwen's cross-layer value cache. Only full-attention candidates are attempted and the first strict pass is kept.

The supplied log failed four quality gates and showed a slowdown (0.57×). These changes correct implementation mismatches; a fresh GPU run is required to establish acceptance and speed. The checkpoint temperature warning is separate from attention transfer.


## 1. Setup
A T4/L4/A100 runtime is recommended. The notebook installs the latest Laya source plus this TinyCeNN-LM repository.

Setup fetches the matching repair branch and clears stale TinyCeNN imports. It refuses to overwrite tracked local edits.


In [ ]:
import os, sys, subprocess, pathlib, importlib, py_compile
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Works in Colab, Kaggle, and ordinary Jupyter kernels.
if pathlib.Path("/content").exists():
    WORK = pathlib.Path("/content")
elif pathlib.Path("/kaggle/working").exists():
    WORK = pathlib.Path("/kaggle/working")
else:
    WORK = pathlib.Path.cwd()

REPO_REF = "codex/fix-laya-pdelta-local32" #@param {type:"string"}
REPO = WORK / "TinyCeNN-LM"
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)])
# Refuse to overwrite local tracked edits; fetch the notebook's matching runtime.
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
    raise RuntimeError("Save/commit tracked changes in TinyCeNN-LM before updating setup.")
subprocess.check_call(["git", "-C", str(REPO), "fetch", "-q", "origin", REPO_REF])
subprocess.check_call(["git", "-C", str(REPO), "checkout", "-q", "--detach", "FETCH_HEAD"])
print("Runtime commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip())

# IMPORTANT: install with this notebook kernel's Python, not a possibly different `pip` executable.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/NandhaKishorM/laya.git", "datasets", "pandas", "pyarrow", "safetensors", "huggingface_hub"])

# Editable-install fallback for notebook environments that cache import paths.
SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# Compile only the modules required by this PDelta3 notebook. An unrelated
# experimental module elsewhere in laya_lab must not block this setup.
LAB_SRC = REPO / "src" / "tinycenn_lm" / "laya_lab"
REQUIRED = [
    "__init__.py", "core.py", "data.py", "evaluate.py", "factory.py",
    "integrated.py", "fusion.py", "pdelta.py", "pdelta_optimized.py",
    "train.py", "runner.py",
]
for name in REQUIRED:
    py_compile.compile(str(LAB_SRC / name), doraise=True)
print("PDelta3 syntax preflight: OK")

# A pull/install alone does not refresh modules already imported by this kernel.
for module_name in list(sys.modules):
    if module_name == "tinycenn_lm" or module_name.startswith("tinycenn_lm."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import tinycenn_lm
from tinycenn_lm.laya_lab import LayaLabConfig, run_experiment
import torch, json, pandas as pd
print("TinyCeNN-LM:", pathlib.Path(tinycenn_lm.__file__).resolve())
print("torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. Experiment configuration

Default model: `convaiinnovations/laya`. Balanced uses 500 steps. Local transfer keeps the high LR, while functional refinement uses a protected low-LR band and ramps the decision loss gradually; the core still finishes at **0.001** and QKV/Wo use 10% of the core LR. This avoids the observed regression when functional distillation was switched on too early at high LR. Rerun setup to refresh imported modules and verify the printed commit/configuration.

Local32 starts with a 0.72 gate. Its 32-token bidirectional window includes self, 15 left and 16 right neighbours. The strict gates remain unchanged. Extended mode increases training to 700 steps; neither mode guarantees acceptance.


In [ ]:
MODEL_ID = "convaiinnovations/laya" #@param ["convaiinnovations/laya", "convaiinnovations/laya-typed-decisions"]
MODE = "balanced" #@param ["smoke", "balanced", "extended"]

TRAIN_STEPS = {"smoke": 120, "balanced": 500, "extended": 900}[MODE]

cfg = LayaLabConfig(
    architecture="pdelta3_gdn2_clvr",
    model_id=MODEL_ID,
    mode=MODE,
    seed=2026,
    feature_dim=128,
    local_kernel=5,
    pdelta_conv_kernel=4,
    pdelta_chunk_size=64,
    pdelta_local_window=32,      # actual bidirectional local softmax, not convolution
    pdelta_local_gate_init=0.72,  # Qwen recipe's local-dominant warm start
    pdelta_train_qkv=True,       # QKV/Wo tune at 0.1x core LR with FP32 master weights
    learning_rate=1e-2,          # START LR: 0.010
    final_learning_rate=1e-3,    # END LR:   0.001
    weight_decay=1e-4,
    training_steps=TRAIN_STEPS,
    # Keep the quality gates strict.
    min_teacher_agreement=0.95,
    max_mean_kl=0.05,
    max_accuracy_drop=0.02,
    max_local_nmse=0.30,
    min_local_cosine=0.88,
    output_dir="/content/laya_tinycenn",
)
assert cfg.pdelta_local_window == 32
print(cfg)


## 3. Local transfer and functional refinement

- Batched Local32 SDPA uses overlapping windows, avoiding a dense T×T local score matrix.
- Padding is masked before convolution to prevent contamination of valid tokens.
- Local transfer occupies about 60% of training by default. Functional refinement starts earlier only when the held-out local probe is already close to the strict local gates.
- Functional refinement drops into a protected low-LR band, ramps decision/action/logit losses gradually, and uses tighter gradient clipping so it cannot erase the local solution.
- FP16 training uses gradient scaling; trainable projections keep FP32 master weights.
- Fit/probe are split by state before constructing questions. Repeated fast checks use held-out training states, not the final test set.
- All 24 local probe sequences are evaluated in small batches. Checkpoint ranking prioritizes probe thresholds, then aggregate error; the restored best checkpoint must still pass every strict gate.
- Rejected best checkpoints are saved before restoring the original layer.

The GDN2 recurrence still uses a per-token loop. Measure actual end-to-end speed; Local32 alone does not guarantee a faster model.


In [ ]:
teacher, student, report = run_experiment(cfg)


## 4. Laya-native result summary and fast teacher comparison

The gold-label evaluation remains the authoritative result. `fast_eval` is a separate batched, label-free teacher/student comparison that is cheap enough to run repeatedly and reports agreement, KL, probability drift, centered-logit error, and forward speed.


In [ ]:
converted = bool(report.get("accepted_layers"))
student_label = report["architecture"] if converted else report["architecture"] + " (no layer accepted; teacher restored)"

summary = pd.DataFrame([
    {
        "model": "Laya teacher",
        **{k: report["teacher_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
    },
    {
        "model": student_label,
        **{k: report["student_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
        "teacher_agreement": report["student_final"].get("teacher_agreement"),
        "teacher_KL": report["student_final"].get("mean_teacher_kl"),
    },
])
display(summary)

print("Conversion succeeded:", converted)
print("Accepted attention layers:", report["accepted_layers"])
print("Training steps/candidate:", report.get("training_steps_per_candidate"))
print("Replacement trainable parameters:", f'{report["replacement_trainable_parameters"]:,}')
print("Gate/final disjoint:", report.get("gate_final_disjoint"))
print("Latency:", json.dumps(report["latency"], indent=2))

print("\nFAST TEACHER ↔ STUDENT COMPARISON")
print("Fast-eval subject:", report.get("fast_eval_subject"))
display(pd.DataFrame([report["fast_eval"]]))
if report.get("restored_teacher_fast_eval") is not None:
    print("Restored-teacher identity check (not the candidate benchmark):")
    display(pd.DataFrame([report["restored_teacher_fast_eval"]]))

if not converted:
    print("NOTE: No replacement passed every strict gate; the final student was restored to the teacher.")
    print("Inspect the candidate table below: it preserves each candidate's best fast/local metrics before restoration.")

for primitive, metrics in report["student_final"]["by_type"].items():
    print(primitive, metrics)


## 5. Re-run your Laya example on the adapted model
The same 'Router' API is preserved. We attach the already-loaded adapted Agent so there is no second model copy. The architecture here replaces the **English Laya** encoder only; Laya's multilingual route can remain the original multilingual checkpoint.


In [ ]:
from laya import Router

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else"
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"]
    },
    "churn_risk": {"type": "noul", "instructions": "Does the user threaten to cancel or leave?"},
    "refund_requested": {"type": "noul", "instructions": "Does the user explicitly request a refund?"}
}

route_name = "typed-decisions" if "typed-decisions" in MODEL_ID else "english"
router = Router(preload=False)
router.attach(route_name, student)
res = router.predict(state, questions, model=route_name)

print("Department       :", res["answers"]["department"]["choice"])
print("Urgency score    :", res["answers"]["urgency"]["score"])
print("Churn risk       :", res["answers"]["churn_risk"]["noul"])
print("Refund requested :", res["answers"]["refund_requested"]["noul"])
print("Routing          :", res["routing"]["model"])
print("\nTeacher/student raw demo comparison:")
print(json.dumps(report["demo"], indent=2, ensure_ascii=False))


## 6. Inspect best candidate checkpoints

This table shows the **best checkpoint reached for each attempted candidate**, including the fast functional comparison. If a candidate is rejected and the final student is restored, these rows still show how close that candidate actually came.


In [ ]:
rows = []
for h in report["history"]:
    fast = h.get("fast_eval", {})
    rows.append({
        "layer": h["layer"],
        "attention_type": h["attention_type"],
        "accepted": h["accepted"],
        "best_step": h.get("best_step"),
        "nmse": h["local"]["nmse"],
        "cosine": h["local"]["cosine"],
        "core_nmse": h["local"].get("core_nmse"),
        "core_cosine": h["local"].get("core_cosine"),
        "fast_agreement": fast.get("teacher_student_top1_agreement"),
        "fast_KL": fast.get("mean_teacher_kl"),
        "fast_prob_L1": fast.get("mean_probability_l1"),
        "fast_speedup": fast.get("forward_speedup_vs_teacher"),
        "gate_agreement": h["gate"].get("teacher_agreement"),
        "gate_KL": h["gate"].get("mean_teacher_kl"),
        "accuracy": h["gate"].get("accuracy"),
        "accuracy_drop": h["accuracy_drop"],
    })
display(pd.DataFrame(rows))


## 7. Saved local outputs

The run writes the accepted adapter (or an empty diagnostic adapter when nothing passes), a JSON report, and per-candidate diagnostic checkpoints under `/content/laya_tinycenn/pdelta3_gdn2_clvr/`.


In [ ]:
from pathlib import Path
out = Path(cfg.output_dir) / cfg.architecture
print("Adapter:", out / "adapter.pt")
print("Report :", out / "report.json")
print((out / "report.json").read_text()[:4000])


## 8. Save the successful adapter to Hugging Face

The cell below creates a small Hugging Face model repository containing `adapter.pt`, `adapter_config.json`, `report.json`, and a generated model card. Put an `HF_TOKEN` secret in Colab (recommended) or let the Hugging Face login widget authenticate you.

By default it **does not publish a failed conversion**: if no attention layer passed the strict gates, it keeps the diagnostics local instead of uploading an adapter that is just the restored teacher.


In [ ]:
import os, json, shutil
from pathlib import Path
from huggingface_hub import HfApi, notebook_login

HF_REPO_ID = "" #@param {type:"string"}
HF_PRIVATE = True #@param {type:"boolean"}
PUSH_FAILED_DIAGNOSTICS = False #@param {type:"boolean"}

out = Path(cfg.output_dir) / cfg.architecture
converted = bool(report.get("conversion_succeeded"))

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

if not token:
    print("HF_TOKEN not found; opening Hugging Face login.")
    notebook_login()

api = HfApi(token=token)
who = api.whoami()
username = who.get("name") or who.get("fullname")
if not username:
    raise RuntimeError("Could not determine Hugging Face account name.")

repo_id = HF_REPO_ID.strip() or f"{username}/laya-pdelta3-gdn2-clvr"
export_dir = Path("/content/laya_pdelta3_hf_export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)

shutil.copy2(out / "report.json", export_dir / "report.json")
if converted:
    shutil.copy2(out / "adapter.pt", export_dir / "adapter.pt")

adapter_cfg = {
    "format": "tinycenn-laya-attention-lab-v1",
    "base_model": cfg.model_id,
    "architecture": cfg.architecture,
    "accepted_layers": report.get("accepted_layers", []),
    "conversion_succeeded": converted,
    "feature_dim": cfg.feature_dim,
    "local_kernel": cfg.local_kernel,
    "pdelta_conv_kernel": cfg.pdelta_conv_kernel,
    "pdelta_chunk_size": cfg.pdelta_chunk_size,
    "learning_rate_start": cfg.learning_rate,
    "learning_rate_end": cfg.final_learning_rate,
    "global_linear_attention": True,
    "single_recurrent_scan": True,
    "autocast_core": True,
    "qkv_frozen": not cfg.pdelta_train_qkv,
    "local_window": cfg.pdelta_local_window,
    "local_gate_init": cfg.pdelta_local_gate_init,
    "hybrid_attention": True,
    "output_projection_calibrated": True,
}
(export_dir / "adapter_config.json").write_text(
    json.dumps(adapter_cfg, indent=2), encoding="utf-8"
)

fast = report.get("fast_eval", {})
student_final = report.get("student_final", {})
card = f"""---
base_model: {cfg.model_id}
tags:
- laya
- tinycenn
- pdelta3
- linear-attention
- knowledge-distillation
---

# Laya PDelta3-GDN2-CLVR Adapter

Base model: `{cfg.model_id}`

Conversion succeeded: **{converted}**

Accepted attention layers: `{report.get("accepted_layers", [])}`

This hybrid adapter uses PDelta3-GDN2 with bidirectional Local32 softmax,
a learned global linear-attention path, encoder-state routing (not Qwen cross-layer V),
and low-LR QKV/output-projection calibration. Training combines attention/core
reconstruction with end-to-end Laya decision distillation.

## Fast teacher/student comparison

- top-1 agreement: {fast.get("teacher_student_top1_agreement")}
- teacher KL: {fast.get("mean_teacher_kl")}
- probability L1: {fast.get("mean_probability_l1")}
- forward speedup vs teacher: {fast.get("forward_speedup_vs_teacher")}
- fast-eval subject: {report.get("fast_eval_subject")}

## Held-out Laya evaluation

- accuracy: {student_final.get("accuracy")}
- teacher agreement: {student_final.get("teacher_agreement")}
- teacher KL: {student_final.get("mean_teacher_kl")}

See `report.json` for the complete metrics, gate history, and configuration.
"""
(export_dir / "README.md").write_text(card, encoding="utf-8")

if not converted and not PUSH_FAILED_DIAGNOSTICS:
    print("HF upload skipped: no replacement passed all strict gates.")
    print("Set PUSH_FAILED_DIAGNOSTICS=True only if you want to upload the report without an adapter.")
else:
    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        private=HF_PRIVATE,
        exist_ok=True,
    )
    api.upload_folder(
        folder_path=str(export_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload optimized Laya PDelta3 adapter and evaluation",
    )
    print("Uploaded to Hugging Face:", f"https://huggingface.co/{repo_id}")
